In [ ]:
! pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu118
! pip install transformers==4.46.3 torch tokenizers==0.20.3 PyMuPDF img2pdf einops easydict addict Pillow numpy
! pip install peft trl datasets accelerate bitsandbytes

In [ ]:
! pip install flash-attn==2.7.3 --no-build-isolation -v
! pip install transformers==4.47.1
! pip install fastapi uvicorn python-multipart pyngrok nest-asyncio
! pip install openai sentencepiece "pydantic<=2.12.3"

In [ ]:
# ── CELL 1 — Import ──────────────────────────────────────────────
import os, json, re, time, glob, shutil, io, base64, threading
from typing import Any, Dict, List, Optional, Tuple

import torch
import cv2
import numpy as np
import nest_asyncio
import uvicorn
from PIL import Image

from peft import PeftModel
from transformers import (
    AutoModel, AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from pydantic import BaseModel, Field
from openai import OpenAI
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok
from google.colab.patches import cv2_imshow

In [ ]:
# ── CELL 2 — Config & API keys ───────────────────────────────────
DEEPSEEK_API_KEY  = "sk-xxxx".strip()
QWEN_MODEL_ID     = "Qwen/Qwen2.5-Math-7B-Instruct"
DEEPSEEK_MODEL    = "deepseek-chat"
OCR_ADAPTER_PATH  = "blue7012/deepseek_ocr_lora"
OCR_MODEL_NAME    = "deepseek-ai/DeepSeek-OCR-2"

!ngrok config add-authtoken 3xxxx

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())

In [ ]:
# ── CELL 3 — Load OCR model (DeepSeek-OCR-2 + LoRA) ─────────────
ocr_model = AutoModel.from_pretrained(
    OCR_MODEL_NAME,
    _attn_implementation="eager",
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    trust_remote_code=True,
    use_safetensors=True,
)
ocr_tokenizer = AutoTokenizer.from_pretrained(OCR_MODEL_NAME, trust_remote_code=True)
ocr_model = PeftModel.from_pretrained(ocr_model, OCR_ADAPTER_PATH)
ocr_model.eval()

ocr_model.generation_config.max_new_tokens    = 2048
ocr_model.generation_config.do_sample         = False
ocr_model.generation_config.repetition_penalty = 1.3
ocr_model.generation_config.no_repeat_ngram_size = 5

print("✅ OCR model loaded")

In [ ]:
# ── CELL 4 — Load Qwen grading model ─────────────────────────────
_quant_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID, use_fast=True)
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_ID,
    device_map="auto",
    quantization_config=_quant_cfg,
    torch_dtype=torch.float16,
)
qwen_model.eval()

deepseek_client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")
# ping
deepseek_client.chat.completions.create(
    model=DEEPSEEK_MODEL,
    messages=[{"role": "user", "content": "ping"}],
    max_tokens=5, temperature=0.0,
)
print("✅ Qwen grading model loaded")
print("✅ DeepSeek client initialized")

In [ ]:
# ── CELL 5 — OCR helpers ─────────────────────────────────────────

def clean_markdown(raw_text: str) -> str:
    text = re.sub(r'<\|ref\|>.*?<\|/ref\|>', '', raw_text)
    text = re.sub(r'<\|det\|>.*?<\|/det\|>', '', text)
    text = re.sub(r'\n\s*\n', '\n\n', text)
    return text.strip()

def resize_for_ocr(img_bgr: np.ndarray, max_side: int = 2048) -> np.ndarray:
    """
    Resize ảnh sao cho cạnh dài nhất <= max_side.
    Giúp model thấy rõ hơn mà không tăng số patch.
    """
    h, w  = img_bgr.shape[:2]
    scale = max_side / max(h, w)
    if scale >= 1.0:
        return img_bgr
    new_w = int(w * scale)
    new_h = int(h * scale)
    return cv2.resize(img_bgr, (new_w, new_h), interpolation=cv2.INTER_AREA)

def process_imagex2(img_bgr, cfg):
    """Xóa đường kẻ vở và vết gạch dày."""
    h_img, w_img = img_bgr.shape[:2]
    total_px = w_img * h_img
    gray   = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    binary = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV,
        blockSize=cfg["block_size"], C=cfg["c_value"],
    )
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary)
    img_clean    = img_bgr.copy()
    min_area_px  = int(cfg["min_area_pct"]    / 100 * total_px)
    area_thresh  = int(cfg["area_thresh_pct"] / 100 * total_px)

    for i in range(1, num_labels):
        x, y, w, h, area = stats[i, :5]
        ar = round(w / h, 2) if h > 0 else 0
        if area < min_area_px:
            continue
        is_notebook_line  = w > (0.6 * w_img)
        is_messy_scribble = h > 15 and area > 800
        if is_notebook_line or is_messy_scribble:
            p = cfg["padding"]
            cv2.rectangle(img_clean,
                (max(x-p, 0),       max(y-p, 0)),
                (min(x+w+p, w_img), min(y+h, h_img)),
                (255, 255, 255), -1)
    return img_clean


def detect_strikethrough(binary, w_img, h_img, cfg):
    """Phát hiện gạch xóa 1 lần qua chữ, tránh nhầm dấu phân số/trừ."""
    kernel_h = cv2.getStructuringElement(
        cv2.MORPH_RECT, (cfg.get("strike_min_width", 18), 1)
    )
    h_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel_h)
    num_h, _, stats_h, _ = cv2.connectedComponentsWithStats(h_lines)
    mask = np.zeros((h_img, w_img), dtype=np.uint8)

    for i in range(1, num_h):
        x, y, w, h, area = stats_h[i, :5]
        ar  = w / h if h > 0 else 0
        min_w = cfg.get("strike_min_width", 18)
        max_w = int(0.55 * w_img)
        if not (min_w < w < max_w and ar > 4 and h <= 6):
            continue

        scan_gap, scan_h = 8, 20
        region_above = binary[max(y-scan_gap-scan_h,0):max(y-scan_gap,0), x:x+w]
        region_below = binary[min(y+h+scan_gap,h_img):min(y+h+scan_gap+scan_h,h_img), x:x+w]

        d_above = region_above.sum() / (region_above.size + 1e-6)
        d_below = region_below.sum() / (region_below.size + 1e-6)

        is_fraction = d_above > 0.08 and d_below > 0.08
        is_minus    = d_above < 0.05 and d_below < 0.05
        if is_fraction or is_minus:
            continue

        p_y = cfg.get("strike_pad_y", 13)
        p_x = cfg.get("strike_pad_x", 4)
        cv2.rectangle(mask,
            (max(x-p_x, 0),       max(y-p_y, 0)),
            (min(x+w+p_x, w_img), min(y+h+p_y, h_img)),
            255, -1)
    return mask


def process_imagex4(img_bgr, cfg):
    """Pass 1: đường kẻ/gạch dày. Pass 2: strikethrough."""
    h_img, w_img = img_bgr.shape[:2]
    gray   = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    binary = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV,
        blockSize=cfg["block_size"], C=cfg["c_value"],
    )

    # Pass 1
    line_mask = np.zeros((h_img, w_img), dtype=np.uint8)
    num_labels, _, stats, _ = cv2.connectedComponentsWithStats(binary)
    total_px = w_img * h_img
    min_area = int(cfg["min_area_pct"]    / 100 * total_px)
    area_thr = int(cfg["area_thresh_pct"] / 100 * total_px)
    for i in range(1, num_labels):
        x, y, w, h, area = stats[i, :5]
        if area < min_area:
            continue
        if w > (0.6 * w_img) or (h > 15 and area > 800):
            p = cfg["padding"]
            cv2.rectangle(line_mask,
                (max(x-p,0),       max(y-p,0)),
                (min(x+w+p,w_img), min(y+h+p,h_img)),
                255, -1)

    img_clean = cv2.inpaint(img_bgr, line_mask, inpaintRadius=5, flags=cv2.INPAINT_TELEA)

    # Pass 2
    gray2   = cv2.cvtColor(img_clean, cv2.COLOR_BGR2GRAY)
    binary2 = cv2.adaptiveThreshold(
        gray2, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV,
        blockSize=cfg["block_size"], C=cfg["c_value"],
    )
    strike_mask = detect_strikethrough(binary2, w_img, h_img, cfg)
    if cv2.countNonZero(strike_mask) > 0:
        img_clean = cv2.inpaint(img_clean, strike_mask, inpaintRadius=7, flags=cv2.INPAINT_TELEA)

    return img_clean

# ════════════════════════════════════════════════════════════════════
# PREPROCESS & MULTI-PAGE HELPERS
# ════════════════════════════════════════════════════════════════════

def enhance_for_ocr(img_bgr: np.ndarray) -> np.ndarray:
    h, w = img_bgr.shape[:2]

    if min(h, w) < 1000:
        scale   = 1000 / min(h, w)
        img_bgr = cv2.resize(img_bgr, None, fx=scale, fy=scale,
                             interpolation=cv2.INTER_CUBIC)

    lab     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l       = clahe.apply(l)
    img_bgr = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)

    blur    = cv2.GaussianBlur(img_bgr, (0, 0), 2)
    img_bgr = cv2.addWeighted(img_bgr, 1.4, blur, -0.4, 0)

    return img_bgr


def detect_spine_column(img_bgr: np.ndarray) -> int:
    """
    Tự động tìm cột gáy sách trong ảnh 2 trang mở.
    Gáy sách thường là vùng tối nhất ở giữa ảnh (30%–70% chiều ngang).
    Trả về index cột chia 2 trang.
    """
    h, w      = img_bgr.shape[:2]
    gray      = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    col_dark  = 255 - gray.mean(axis=0)          # tối = giá trị cao

    mid_start = int(w * 0.30)
    mid_end   = int(w * 0.70)
    profile   = col_dark[mid_start:mid_end]

    # Smooth để tránh nhiễu
    kernel    = np.ones(60) / 60
    smoothed  = np.convolve(profile, kernel, mode='same')

    spine_col = mid_start + int(np.argmax(smoothed))
    return spine_col


def is_two_page_spread(img_bgr: np.ndarray, ratio_threshold: float = 1.2) -> bool:
    """
    Phát hiện ảnh có phải 2 trang mở không dựa vào:
    1. Tỉ lệ chiều rộng/cao > threshold (ảnh ngang rộng)
    2. Có vùng tối ở giữa (gáy sách)
    """
    h, w  = img_bgr.shape[:2]
    ratio = w / h

    if ratio < ratio_threshold:
        return False

    gray      = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    mid_start = int(w * 0.35)
    mid_end   = int(w * 0.65)
    mid_strip = gray[:, mid_start:mid_end]
    mid_mean  = float(mid_strip.mean())

    full_mean = float(gray.mean())

    return (full_mean - mid_mean) > 15


def split_two_page_spread(img_bgr: np.ndarray) -> tuple:
    """
    Tách ảnh 2 trang mở thành (trang trái, trang phải).
    Tự động detect gáy sách, crop sát vào nội dung.
    Returns: (left_page, right_page) — cả 2 đều đã enhance
    """
    spine     = detect_spine_column(img_bgr)
    h, w      = img_bgr.shape[:2]

    margin    = int(w * 0.01)
    left_end  = max(0, spine - margin)
    right_start = min(w, spine + margin)

    left_page  = img_bgr[:, :left_end]
    right_page = img_bgr[:, right_start:]

    left_page  = enhance_for_ocr(left_page)
    right_page = enhance_for_ocr(right_page)

    return left_page, right_page


def prepare_images_for_ocr(img_bgr: np.ndarray) -> list:
    """
    Entry point chính: nhận 1 ảnh bất kỳ, trả về list các ảnh đã xử lý.
    - Ảnh 1 trang → [enhanced_image]
    - Ảnh 2 trang mở → [left_enhanced, right_enhanced]
    Tất cả đều đã qua enhance_for_ocr.
    """
    if is_two_page_spread(img_bgr):
        left, right = split_two_page_spread(img_bgr)
        return [left, right]
    else:
        return [enhance_for_ocr(img_bgr)]


In [ ]:
# ── CELL 6 — OCR post-process (DeepSeek) ─────────────────────────

def process_and_structure_ocr_with_deepseek(ocr_text: str) -> dict:
    """Sửa lỗi chính tả tiếng Việt + tách JSON câu/ý."""
    client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")
    system_prompt = """Bạn là một hệ thống AI xử lý dữ liệu bài tập toán học.
Văn bản đầu vào là kết quả OCR bài tập toán, có thể chứa lỗi nhận diện nét chữ.
Nhiệm vụ:
1. Sửa từ tiếng Việt sai chính tả, sai ngữ cảnh.
2. TUYỆT ĐỐI GIỮ NGUYÊN công thức LaTeX (\[ \], \( \), $$ $$) và TUYỆT ĐỐI GIỮ NGUYÊN KẾT QUẢ TOÁN HỌC KỂ CẢ KẾT QUẢ SAI.
3. Nhận diện các ý nhỏ a), b), c) của BÀI 1 được truyền vào.
4. TRẢ VỀ JSON. Không giải thích thêm.
5. CHÚ Ý: Bài được truyền vào chỉ là Bài 1

Format JSON:
{
  "questions": [
    {
      "question_number": "Bài 1",
      "main_content": "...",
      "sub_parts": [
        {"part": "a", "content": "..."},
        {"part": "b", "content": "..."}
      ]
    }
  ]
}"""
    try:
        resp = client.chat.completions.create(
            model=DEEPSEEK_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": ocr_text},
            ],
            temperature=0.1,
            response_format={"type": "json_object"},
        )
        return json.loads(resp.choices[0].message.content)
    except Exception as e:
        print(f"❌ OCR post-process error: {e}")
        return {"error": str(e), "raw_text": ocr_text}

In [ ]:
# ── CELL 7 — Grading schemas (Pydantic) ──────────────────────────

class RubricItemEn(BaseModel):
    criterion_id: str
    description:  str
    max_score:    float

class AlignedPartVI(BaseModel):
    question_number:          str
    question_problem_vi:      str
    part:                     str
    part_max_score:           float
    question_max_score:       float = 0.0
    part_problem_vi:          str
    teacher_solution_steps_vi: List[str]
    rubric_vi:                List[Dict[str, Any]]
    student_answer_vi:        str

class AlignedPartEN(BaseModel):
    question_number:           str
    part:                      str
    question_problem_en:       str
    part_problem_en:           str
    teacher_solution_steps_en: List[str]
    rubric_en:                 List[RubricItemEn]
    student_answer_en:         str
    part_max_score:            float = 0.0

class CriterionScore(BaseModel):
    criterion_id:  str
    awarded_score: float
    max_score:     float
    reason:        str

class RepairedSubPart(BaseModel):
    question:    str
    part:        str
    criteria:    List[CriterionScore]
    explanation: str

class RepairedBatch(BaseModel):
    items: List[RepairedSubPart]

In [ ]:
# ── CELL 8 — Grading helpers ─────────────────────────────────────

def _safe_text(v) -> str:
    return "" if v is None else str(v).strip()

def _to_float_safe(v, default=0.0) -> float:
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        m = re.search(r"-?\d+(?:\.\d+)?", v.strip().replace(",", ""))
        if m:
            return float(m.group(0))
    return default

def _sum_rubric(rubric) -> float:
    return round(sum(float(x.get("max_score", 0)) for x in (rubric or [])), 4)

def get_part_max_score(subpart: dict) -> float:
    rules = subpart.get("score_rules", {}) or {}
    if rules.get("use_rubric_sum_as_part_max", True):
        return _sum_rubric(subpart.get("rubric", []))
    explicit = float(subpart.get("part_max_score", 0) or 0)
    return explicit if explicit > 0 else _sum_rubric(subpart.get("rubric", []))

def validate_and_normalize_exam_blueprint(bp: dict) -> dict:
    bp = json.loads(json.dumps(bp))
    for q in bp.get("questions", []):
        derived_q_max = 0.0
        for sp in q.get("sub_parts", []):
            rubric_sum    = _sum_rubric(sp.get("rubric", []))
            derived_p_max = get_part_max_score(sp)
            if "part_max_score" in sp:
                explicit = float(sp.get("part_max_score", 0) or 0)
                if abs(explicit - derived_p_max) > 1e-9:
                    raise ValueError(
                        f"Lệch điểm {q.get('question_number')} phần {sp.get('part')}: "
                        f"part_max_score={explicit} vs rubric_sum={rubric_sum}"
                    )
            sp["part_max_score"] = derived_p_max
            derived_q_max += derived_p_max
        derived_q_max = round(derived_q_max, 4)
        if "question_max_score" in q:
            explicit = float(q.get("question_max_score", 0) or 0)
            if abs(explicit - derived_q_max) > 1e-9:
                raise ValueError(
                    f"Lệch tổng {q.get('question_number')}: "
                    f"question_max_score={explicit} vs sum_parts={derived_q_max}"
                )
        q["question_max_score"] = derived_q_max
    return bp

def build_student_part_map(student_input: dict) -> dict:
    result = {}
    for q in (student_input.get("final_markdown", {}) or {}).get("questions", []):
        qn = _safe_text(q.get("question_number"))
        for sp in q.get("sub_parts", []) or []:
            result[(qn, _safe_text(sp.get("part")))] = _safe_text(sp.get("content"))
    return result

def build_aligned_part_items(
    exam_blueprint: Dict[str, Any],
    student_input: Dict[str, Any]
) -> List[AlignedPartVI]:
    exam_blueprint = validate_and_normalize_exam_blueprint(exam_blueprint)

    student_questions = (
        student_input.get("final_markdown", {}).get("questions", [])
        if isinstance(student_input, dict) else []
    )

    items: List[AlignedPartVI] = []

    # Fallback: nếu chỉ có 1 question bên student, align theo part
    single_student_question = student_questions[0] if len(student_questions) == 1 else None
    single_student_part_map = {}
    if single_student_question:
        for sp in single_student_question.get("sub_parts", []) or []:
            single_student_part_map[_safe_text(sp.get("part"))] = _safe_text(sp.get("content"))

    student_map = build_student_part_map(student_input)

    for q in exam_blueprint.get("questions", []):
        qn = _safe_text(q.get("question_number"))
        question_problem_vi = _safe_text(q.get("problem"))
        question_max_score = float(q.get("question_max_score", 0.0))

        for sp in q.get("sub_parts", []):
            part = _safe_text(sp.get("part"))

            # ưu tiên match exact theo (question_number, part)
            student_answer_vi = student_map.get((qn, part), "")

            # fallback nếu không khớp question_number nhưng chỉ có 1 câu sinh viên
            if not student_answer_vi and single_student_question is not None:
                student_answer_vi = single_student_part_map.get(part, "")

            items.append(
                AlignedPartVI(
                    question_number=qn,
                    question_problem_vi=question_problem_vi,
                    part=part,
                    part_max_score=get_part_max_score(sp),
                    question_max_score=question_max_score,
                    part_problem_vi=_safe_text(sp.get("problem")),
                    teacher_solution_steps_vi=[
                        _safe_text(x) for x in (sp.get("teacher_solution_steps") or [])
                    ],
                    rubric_vi=sp.get("rubric", []) or [],
                    student_answer_vi=student_answer_vi,
                )
            )

    return items

def build_blueprint_part_map(exam_bp: dict) -> dict:
    exam_bp = validate_and_normalize_exam_blueprint(exam_bp)
    result  = {}
    for q in exam_bp.get("questions", []):
        qn = _safe_text(q.get("question_number"))
        for sp in q.get("sub_parts", []):
            result[(qn, _safe_text(sp.get("part")))] = sp
    return result

def _chunk_list(items, n):
    return [items[i:i+n] for i in range(0, len(items), n)]

In [ ]:
# ── CELL 9 — DeepSeek grading helpers ────────────────────────────

def deepseek_generate_with_retry(
    contents: str,
    temperature: float = 0.0,
    max_tokens: int = 8192,
    response_format=None,
    max_retries: int = 5,
    base_sleep: float = 4.0,
):
    last_err = None
    for attempt in range(max_retries):
        try:
            kwargs = dict(
                model=DEEPSEEK_MODEL,
                messages=[{"role": "user", "content": contents}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            if response_format:
                kwargs["response_format"] = response_format
            return deepseek_client.chat.completions.create(**kwargs)
        except Exception as e:
            last_err = e
            msg = str(e).lower()
            retryable = any(x in msg for x in ["429","rate limit","quota","timeout","502","503","504"])
            if not retryable:
                raise
            sleep_s = base_sleep * (2 ** attempt)
            print(f"[DeepSeek retry {attempt+1}] sleep {sleep_s:.1f}s")
            time.sleep(sleep_s)
    raise last_err


def translate_aligned_parts_with_deepseek(
    aligned_items_vi: List[AlignedPartVI],
    batch_size: int = 6,
    debug: bool = False,
) -> List[AlignedPartEN]:
    translated = []
    schema_ex = {"items": [{"question_number": "Bài 1", "part": "a",
        "question_problem_en": "...", "part_problem_en": "...",
        "teacher_solution_steps_en": ["step1"], "part_max_score": 0.5,
        "rubric_en": [{"criterion_id": "R1", "description": "...", "max_score": 0.25}],
        "student_answer_en": "..."}]}

    for batch_idx, batch in enumerate(_chunk_list(aligned_items_vi, batch_size), 1):
        payload = [{"question_number": it.question_number, "part": it.part,
            "question_problem_vi": it.question_problem_vi,
            "part_problem_vi": it.part_problem_vi,
            "part_max_score": it.part_max_score,
            "teacher_solution_steps_vi": it.teacher_solution_steps_vi,
            "rubric_vi": it.rubric_vi,
            "student_answer_vi": it.student_answer_vi} for it in batch]

        prompt = f"""Translate the following Vietnamese math grading items into English.
RULES: Return ONLY JSON with root key "items". Translate natural language only.
Preserve ALL math/LaTeX/symbols exactly. Keep question_number, part, part_max_score unchanged.
Format: {json.dumps(schema_ex, ensure_ascii=False)}
Input: {json.dumps(payload, ensure_ascii=False, indent=2)}""".strip()

        resp = deepseek_generate_with_retry(
            contents=prompt, temperature=0.0, max_tokens=8192,
            response_format={"type": "json_object"},
        )
        raw  = resp.choices[0].message.content
        data = json.loads(raw)
        if debug:
            print(f"=== TRANSLATE BATCH {batch_idx} ===")
            print(raw[:800])
        for obj in data.get("items", []):
            en = AlignedPartEN.model_validate(obj)
            # đảm bảo part_max_score được giữ lại
            if en.part_max_score == 0.0:
                matched = next((it for it in batch
                                if it.question_number == en.question_number
                                and it.part == en.part), None)
                if matched:
                    en.part_max_score = matched.part_max_score
            translated.append(en)

    if len(translated) != len(aligned_items_vi):
        raise ValueError(f"Số item dịch không khớp: expected={len(aligned_items_vi)} got={len(translated)}")
    return translated

In [ ]:
# ── CELL 10 — Qwen inference + parse + repair ────────────────────

def build_qwen_prompt_en(item_en: AlignedPartEN) -> str:
    rubric_json  = json.dumps([r.model_dump() for r in item_en.rubric_en], ensure_ascii=False, indent=2)
    steps_json   = json.dumps(item_en.teacher_solution_steps_en, ensure_ascii=False, indent=2)
    return f"""You are grading ONLY sub-part {item_en.question_number} - {item_en.part}.
Grade the student answer AGAINST THE RUBRIC ONLY. Do NOT solve from scratch.
Total sub-part score: {item_en.part_max_score}

Return EXACTLY this format:
R1: <awarded>/<max> - <reason>
...
Explanation: <1-3 sentences>

Problem: {item_en.part_problem_en}
Teacher steps: {steps_json}
Rubric: {rubric_json}
Student answer: {item_en.student_answer_en}""".strip()


def call_qwen(prompt: str, max_new_tokens: int = 768) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a strict math grader. "
                "Respond in English only. "
                "Be concise and follow the requested output format exactly."
            )
        },
        {"role": "user", "content": prompt},
    ]

    text = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = qwen_tokenizer([text], return_tensors="pt").to(qwen_model.device)
    qwen_model.generation_config.max_length = None

    with torch.no_grad():
        output_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id,
            eos_token_id=qwen_tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return qwen_tokenizer.decode(generated, skip_special_tokens=True).strip()


def build_qwen_prompt_en(item_en: AlignedPartEN, part_max_score: float) -> str:
    rubric_json = json.dumps([r.model_dump() for r in item_en.rubric_en], ensure_ascii=False, indent=2)
    teacher_steps_json = json.dumps(item_en.teacher_solution_steps_en, ensure_ascii=False, indent=2)

    return f"""
        You are grading ONLY sub-part {item_en.question_number} - {item_en.part}.

        You must grade the student's answer AGAINST THE RUBRIC ONLY.
        Do NOT solve the problem from scratch unless needed to verify correctness.
        Do NOT award points outside the rubric.

        Return in this exact plain-text format:
        R1: <awarded_score>/<max_score> - <short reason>
        R2: <awarded_score>/<max_score> - <short reason>
        R3: <awarded_score>/<max_score> - <short reason>
        Explanation: <1-3 sentences>

        STRICT SCORING RULES:
        - Score each criterion independently.
        - Awarded score cannot exceed that criterion's max_score.
        - If the student skips a criterion, give 0 for that criterion.
        - The sum of all awarded criterion scores MUST NOT exceed the total sub-part score.
        - Total sub-part score for this part is exactly:
        {part_max_score}
        - IMPORTANT: the total sub-part score should equal the sum of rubric max scores, not any external assumption.
        - The student's answer is EMPTY.
        You must assign 0 to every rubric criterion.

        Return in this exact plain-text format:
        R1: 0/{part_max_score} - Empty student answer
        Explanation: The student's answer is empty, so no rubric criterion can be awarded.
        Problem:
        {item_en.part_problem_en}

        Teacher solution steps:
        {teacher_steps_json}

        Rubric:
        {rubric_json}

        Student answer:
        {item_en.student_answer_en}
        """.strip()


def run_qwen_grading_on_parts(
    aligned_items_en: List[AlignedPartEN],
    debug: bool = False,
) -> List[Dict[str, Any]]:
    results = []
    for item in aligned_items_en:
        prompt     = build_qwen_prompt_en(item)
        raw_output = call_qwen(prompt)
        if debug:
            print(f"=== QWEN | {item.question_number} | {item.part} ===")
            print(raw_output)
        results.append({
            "question":       item.question_number,
            "part":           item.part,
            "part_max_score": item.part_max_score,
            "raw_output":     raw_output,
        })
    return results


def _parse_qwen_raw(raw: str, question: str, part: str) -> dict:
    pattern = re.compile(
        r"^(R\d+)\s*:\s*([0-9]+(?:\.[0-9]+)?)\s*/\s*([0-9]+(?:\.[0-9]+)?)\s*-\s*(.*)$",
        re.IGNORECASE,
    )
    criteria, explanation = [], ""
    for line in [x.strip() for x in raw.splitlines() if x.strip()]:
        if line.upper().startswith("EXPLANATION:"):
            explanation = line.split(":", 1)[1].strip()
            continue
        m = pattern.match(line)
        if m:
            criteria.append({"criterion_id": m.group(1), "awarded_score": float(m.group(2)),
                              "max_score": float(m.group(3)), "reason": m.group(4).strip()})
    return {"question": question, "part": part, "criteria": criteria, "explanation": explanation}


def _needs_repair(parsed: dict) -> bool:
    return not parsed.get("criteria") or not parsed.get("explanation", "").strip()


def repair_qwen_outputs_with_deepseek(
    raw_results: List[Dict[str, Any]],
    debug: bool = False,
) -> RepairedBatch:
    locally_parsed = [_parse_qwen_raw(x["raw_output"], x["question"], x["part"]) for x in raw_results]
    repair_targets = [rr for rr, p in zip(raw_results, locally_parsed) if _needs_repair(p)]
    repaired_map   = {(p["question"], p["part"]): p for p in locally_parsed if not _needs_repair(p)}

    if repair_targets:
        prompt = f"""Convert Qwen grading outputs to JSON. DO NOT grade. DO NOT change scores.
Return ONLY: {{"items":[{{"question":"...","part":"...","criteria":[{{"criterion_id":"R1","awarded_score":0.25,"max_score":0.25,"reason":"..."}}],"explanation":"..."}}]}}
Raw: {json.dumps(repair_targets, ensure_ascii=False, indent=2)}""".strip()
        resp = deepseek_generate_with_retry(
            contents=prompt, temperature=0.0, max_tokens=8192,
            response_format={"type": "json_object"},
        )
        raw  = resp.choices[0].message.content
        data = json.loads(raw)
        if debug:
            print("=== DEEPSEEK REPAIR ===")
            print(raw[:1000])
        for item in data.get("items", []):
            q, p = _safe_text(item.get("question")), _safe_text(item.get("part"))
            repaired_map[(q, p)] = {
                "question": q, "part": p,
                "criteria": [{"criterion_id": _safe_text(c.get("criterion_id")),
                               "awarded_score": _to_float_safe(c.get("awarded_score")),
                               "max_score": _to_float_safe(c.get("max_score")),
                               "reason": _safe_text(c.get("reason"))}
                              for c in item.get("criteria", []) if isinstance(c, dict)],
                "explanation": _safe_text(item.get("explanation")),
            }

    fixed = []
    for rr in raw_results:
        q, p  = rr["question"], rr["part"]
        fixed.append(repaired_map.get((q, p)) or _parse_qwen_raw(rr["raw_output"], q, p))
    return RepairedBatch.model_validate({"items": fixed})

In [ ]:
# ── CELL 11 — Score compute + translate explanation ───────────────

def compute_part_scores(repaired_item: RepairedSubPart, blueprint_sp: dict) -> dict:
    expected_max = get_part_max_score(blueprint_sp)
    rubric_map   = {_safe_text(r["criterion_id"]): float(r["max_score"])
                    for r in blueprint_sp.get("rubric", [])}
    total, normalized = 0.0, []

    for cid, exp_max in rubric_map.items():
        matched = next((c for c in repaired_item.criteria if _safe_text(c.criterion_id) == cid), None)
        if matched is None:
            normalized.append({"criterion_id": cid, "awarded_score": 0.0,
                                "max_score": exp_max, "reason": "Missing."})
            continue
        raw_aw  = float(matched.awarded_score or 0)
        raw_max = float(matched.max_score or 0)
        if raw_max > 0 and abs(raw_max - exp_max) > 1e-9:
            awarded = round(max(0, min(1, raw_aw / raw_max)) * exp_max, 4)
        else:
            awarded = round(max(0, min(raw_aw, exp_max)), 4)
        total += awarded
        normalized.append({"criterion_id": cid, "awarded_score": awarded,
                            "max_score": exp_max, "reason": matched.reason})

    total = round(min(total, expected_max), 4)
    return {"score": total, "final_score": total,
            "expected_max_score": expected_max,
            "deduction": round(expected_max - total, 4),
            "criteria": normalized}


def translate_explanations_to_vi(merged: list, batch_size=12, debug=False) -> dict:
    result, payload = {}, [{"question": it["question"], "part": it["part"],
                             "explanation_en": it.get("explanation_en", "")} for it in merged]
    for bi, batch in enumerate(_chunk_list(payload, batch_size), 1):
        prompt = f"""Translate English grading explanations to natural Vietnamese.
Return ONLY JSON: {{"items":[{{"question":"...","part":"...","explanation_vi":"..."}}]}}
Input: {json.dumps(batch, ensure_ascii=False, indent=2)}""".strip()
        resp = deepseek_generate_with_retry(contents=prompt, temperature=0.0, max_tokens=4096,
                                            response_format={"type": "json_object"})
        raw  = resp.choices[0].message.content
        data = json.loads(raw)
        if debug:
            print(f"=== EXPLAIN VI BATCH {bi} ===")
            print(raw[:600])
        for obj in data.get("items", []):
            result[(_safe_text(obj.get("question")), _safe_text(obj.get("part")))] =                 _safe_text(obj.get("explanation_vi"))
    return result


def merge_grading_results(
    raw_results:    List[Dict[str, Any]],
    repaired_batch: RepairedBatch,
    exam_bp:        dict,
    debug:          bool = False,
) -> list:
    raw_map  = {(x["question"], x["part"]): x["raw_output"] for x in raw_results}
    bp_map   = build_blueprint_part_map(exam_bp)
    merged   = []

    for item in repaired_batch.items:
        bp         = bp_map[(item.question, item.part)]
        score_info = compute_part_scores(item, bp)
        raw_out    = raw_map.get((item.question, item.part), "")
        confidence = 1.0
        if len(raw_out.strip()) < 30:   confidence -= 0.25
        if not item.explanation.strip(): confidence -= 0.20
        if score_info["final_score"] > score_info["expected_max_score"] + 1e-9: confidence -= 0.40
        merged.append({
            "question":          item.question,
            "part":              item.part,
            "score":             score_info["score"],
            "final_score":       score_info["final_score"],
            "expected_max_score":score_info["expected_max_score"],
            "deduction":         score_info["deduction"],
            "criteria":          score_info["criteria"],
            "explanation_en":    item.explanation,
            "explanation_vi":    "",
            "confidence":        round(max(0.0, min(1.0, confidence)), 2),
            "raw_output":        raw_out,
        })

    vi_map = translate_explanations_to_vi(merged, debug=debug)
    for item in merged:
        item["explanation_vi"] = vi_map.get((item["question"], item["part"]), "")
    return merged

def run_qwen_grading_all(
    translated_items: List[AlignedPartEN],
    exam_blueprint: Dict[str, Any],
    debug: bool = True
) -> List[Dict[str, Any]]:
    blueprint_map = build_blueprint_part_map(exam_blueprint)
    raw_results = []

    for item in translated_items:
        bp = blueprint_map[(item.question_number, item.part)]
        rubric = bp.get("rubric", []) or []

        if not str(item.student_answer_en).strip():
            lines = []
            for r in rubric:
                cid = _safe_text(r.get("criterion_id"))
                max_score = float(r.get("max_score", 0.0))
                lines.append(f"{cid}: 0/{max_score} - Empty student answer")
            lines.append("Explanation: The student's answer is empty, so no rubric criterion can be awarded.")

            raw_output = "\n".join(lines)
        else:
            prompt = build_qwen_prompt_en(
                item_en=item,
                part_max_score=float(bp.get("part_max_score", 0.0)),
            )
            raw_output = call_qwen(prompt, max_new_tokens=768)

        if debug:
            print(f"=== RAW QWEN OUTPUT | {item.question_number} | {item.part} ===")
            print(raw_output[:1200])

        raw_results.append({
            "question": item.question_number,
            "part": item.part,
            "raw_output": raw_output,
        })

    return raw_results
def compute_part_scores_from_criteria(
    repaired_item: RepairedSubPart,
    blueprint_subpart: Dict[str, Any]
) -> Dict[str, Any]:
    expected_part_max = get_part_max_score(blueprint_subpart)

    rubric_map = {
        _safe_text(r["criterion_id"]): float(r["max_score"])
        for r in blueprint_subpart.get("rubric", [])
    }

    total_score = 0.0
    normalized_criteria = []

    for criterion_id, expected_max in rubric_map.items():
        matched = None
        for c in repaired_item.criteria:
            if _safe_text(c.criterion_id) == criterion_id:
                matched = c
                break

        if matched is None:
            normalized_criteria.append({
                "criterion_id": criterion_id,
                "awarded_score": 0.0,
                "max_score": expected_max,
                "reason": "Missing in Qwen output."
            })
            continue

        raw_awarded = float(matched.awarded_score or 0.0)
        raw_max = float(matched.max_score or 0.0)

        if raw_max > 0 and abs(raw_max - expected_max) > 1e-9:
            ratio = max(0.0, min(1.0, raw_awarded / raw_max))
            awarded = round(ratio * expected_max, 4)
        else:
            awarded = round(max(0.0, min(raw_awarded, expected_max)), 4)

        total_score += awarded
        normalized_criteria.append({
            "criterion_id": criterion_id,
            "awarded_score": awarded,
            "max_score": expected_max,
            "reason": matched.reason,
        })

    total_score = round(min(total_score, expected_part_max), 4)

    return {
        "score": total_score,
        "deduction": round(expected_part_max - total_score, 4),
        "final_score": total_score,
        "expected_max_score": expected_part_max,
        "criteria": normalized_criteria,
    }

def compute_confidence(raw_text: str, final_score: float, expected_max_score: float, explanation: str) -> float:
    confidence = 1.0

    if len(raw_text.strip()) < 30:
        confidence -= 0.25
    if not explanation.strip():
        confidence -= 0.20
    if final_score < 0 or final_score > expected_max_score + 1e-9:
        confidence -= 0.40

    return max(0.0, min(1.0, confidence))

def merge_repaired_with_raw(
    raw_results: List[Dict[str, Any]],
    repaired_batch: RepairedBatch,
    exam_blueprint: Dict[str, Any],
    debug: bool = True
) -> List[Dict[str, Any]]:
    raw_map = {(x["question"], x["part"]): x["raw_output"] for x in raw_results}
    blueprint_map = build_blueprint_part_map(exam_blueprint)

    merged = []
    for item in repaired_batch.items:
        bp = blueprint_map[(item.question, item.part)]
        score_info = compute_part_scores_from_criteria(item, bp)

        raw_output = raw_map.get((item.question, item.part), "")
        merged.append({
            "question": item.question,
            "part": item.part,
            "score": score_info["score"],
            "deduction": score_info["deduction"],
            "final_score": score_info["final_score"],
            "expected_max_score": score_info["expected_max_score"],
            "criteria": score_info["criteria"],
            "explanation_en": item.explanation,
            "confidence": compute_confidence(
                raw_text=raw_output,
                final_score=score_info["final_score"],
                expected_max_score=score_info["expected_max_score"],
                explanation=item.explanation,
            ),
            "raw_output": raw_output,
            "repair_used": True,
        })

    explanation_map_vi = translate_explanations_to_vi(merged, debug=debug)
    for item in merged:
        item["explanation_vi"] = explanation_map_vi.get((item["question"], item["part"]), "")

    return merged

In [ ]:
# ── CELL 12 — Pipeline chấm điểm ────────────────────────────────
def grade_full_pipeline(
    exam_blueprint: Dict[str, Any],
    student_input: Dict[str, Any],
    debug: bool = True
) -> Dict[str, Any]:
    aligned_items_vi = build_aligned_part_items(exam_blueprint, student_input)

    translated_items = translate_aligned_parts_with_deepseek(
        aligned_items_vi=aligned_items_vi,
        batch_size=6,
        debug=debug,
    )

    raw_results = run_qwen_grading_all(
        translated_items=translated_items,
        exam_blueprint=exam_blueprint,
        debug=debug,
    )

    repaired_batch = repair_qwen_outputs_with_deepseek(
        raw_results=raw_results,
        debug=debug,
    )

    merged = merge_repaired_with_raw(
        raw_results=raw_results,
        repaired_batch=repaired_batch,
        exam_blueprint=exam_blueprint,
        debug=debug,
    )

    total_score = round(sum(x["final_score"] for x in merged), 4)
    total_max_score = round(sum(x["expected_max_score"] for x in merged), 4)

    return {
        "total_score": total_score,
        "total_max_score": total_max_score,
        "parts": merged,
    }


In [ ]:
# ── CELL 13 — FastAPI app ────────────────────────────────────────

app = FastAPI(title="OCR + Grading API")
app.add_middleware(CORSMiddleware,
    allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

DEFAULT_CFG = {
    "block_size": 15, "c_value": 18,
    "min_area_pct": 0.001, "area_thresh_pct": 0.03,
    "ar_thresh": 2, "padding": 6,
    "strike_min_width": 20, "strike_pad_y": 13, "strike_pad_x": 4,
}

# ── /api/v1/clean_image ───────────────────────────────────────────
@app.post("/api/v1/clean_image")
async def clean_image_endpoint(
    files: List[UploadFile] = File(...),
    block_size:      int   = Form(15),
    c_value:         int   = Form(50),
    min_area_pct:    float = Form(0.001),
    area_thresh_pct: float = Form(0.03),
    ar_thresh:       float = Form(2.0),
    padding:         int   = Form(6),
    strike_min_width:int   = Form(20),
    strike_pad_y:    int   = Form(13),
    strike_pad_x:    int   = Form(4),
):

    cfg = {
        "block_size": block_size if block_size % 2 == 1 else block_size + 1,
        "c_value": c_value, "min_area_pct": min_area_pct,
        "area_thresh_pct": area_thresh_pct, "ar_thresh": ar_thresh,
        "padding": padding, "strike_min_width": strike_min_width,
        "strike_pad_y": strike_pad_y, "strike_pad_x": strike_pad_x,
    }
    processed_images = []
    try:
        for i, file in enumerate(files):
            contents = await file.read()
            nparr    = np.frombuffer(contents, np.uint8)
            img_bgr  = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
            img_clean = process_imagex2(img_bgr, cfg)
            _, buf    = cv2.imencode(".png", img_clean)
            cv2_imshow(img_clean)
            processed_images.append({
                "file": i + 1,
                "filename": file.filename,
                "image_base64": base64.b64encode(buf).decode("utf-8")
            })
        return {
            "success":   True,
            "files":  processed_images,
        }
    except Exception as e:
        return {"success": False, "error": str(e)}


# ── /api/v1/ocr ──────────────────────────────────────────────────
@app.post("/api/v1/ocr")
async def ocr_endpoint(
    files: List[UploadFile] = File(...),
    prompt:      str  = Form("<image>\n<|grounding|>Convert the document to markdown."),
    show_labels: bool = Form(True),
):

    output_dir = "./output_api"
    merged_markdown = ""
    temp_paths = []
    bbox_b64s = []
    try:
        for i, file in enumerate(files):
          temp_path = f"temp_ocr_page_{i}_{file.filename}"
          temp_paths.append(temp_path)
          os.makedirs(output_dir, exist_ok=True)
          contents = await file.read()
          nparr    = np.frombuffer(contents, np.uint8)
          img_bgr  = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
          # ── Resize về max 2048px trước khi feed vào model ────────
          img_bgr  = resize_for_ocr(img_bgr, max_side=2048)
          cv2.imwrite(temp_path, img_bgr, [cv2.IMWRITE_JPEG_QUALITY, 95])

          with torch.no_grad():
              ocr_model.infer(
                  ocr_tokenizer,
                  prompt=prompt,
                  image_file=temp_path,
                  output_path=output_dir,
                  base_size=1024,
                  image_size=768,
                  crop_mode=True,
                  save_results=True,
              )

          mmd_files = glob.glob(f"{output_dir}/*.mmd")
          if mmd_files:
              with open(mmd_files[0], "r", encoding="utf-8") as f:
                  raw_text = f.read()
          else:
              return {
                  "success":  False,
                  "filename": file.filename,
                  "error":    "Model không sinh ra file .mmd",
              }
          merged_markdown += f"\n\n--- [Trang {i+1}: {file.filename}] ---\n\n" + raw_text
          bbox_b64 = ""
          if raw_text:
              # Đọc file bbox đã được model lưu sẵn
              bbox_path = os.path.join(output_dir, "result_with_boxes.jpg")
              if os.path.exists(bbox_path):
                  with open(bbox_path, "rb") as f_bbox:
                      bbox_b64 = base64.b64encode(f_bbox.read()).decode("utf-8")
              bbox_b64s.append({
                  "file": i + 1,
                  "filename": file.filename,
                  "image_base64": bbox_b64
              })



        return {
            "success":         True,
            "raw_result":      merged_markdown,
            "markdown_result": clean_markdown(merged_markdown),
            "bbox_images_b64":  bbox_b64s,
        }

    except Exception as e:
        return {"success": False, "error": str(e)}
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)
        if os.path.exists(output_dir):
            shutil.rmtree(output_dir, ignore_errors=True)


# ── /api/v1/fix_vn_and_struct ────────────────────────────────────
@app.post("/api/v1/fix_vn_and_struct")
async def fix_vn_and_struct_endpoint(text: str = Form("")):
    try:
        result = process_and_structure_ocr_with_deepseek(text)
        return {"success": True, "final_markdown": result}
    except Exception as e:
        return {"success": False, "error": str(e)}


# ── /api/v1/grade ────────────────────────────────────────────────
@app.post("/api/v1/grade")
async def grade_endpoint(
    exam_blueprint: str = Form(...),
    student_input:  str = Form(...),
    debug:          bool = Form(False),
):
    """
    Chấm điểm bài toán viết tay.

    Parameters
    ----------
    exam_blueprint : str (JSON)
        Blueprint đề thi gồm câu hỏi, lời giải mẫu, rubric và thang điểm.
    student_input : str (JSON)
        Output từ /api/v1/fix_vn_and_struct — dạng {"final_markdown": {"questions": [...]}}.
    debug : bool
        Nếu True, in log trung gian ra stdout.

    Returns
    -------
    JSON
    {
        "success": true,
        "total_score": 1.5,
        "total_max_score": 2.0,
        "parts": [
            {
                "question": "Bài 1",
                "part": "a",
                "score": 0.75,
                "final_score": 0.75,
                "expected_max_score": 0.75,
                "deduction": 0.0,
                "criteria": [
                    {"criterion_id":"R1","awarded_score":0.25,"max_score":0.25,"reason":"..."}
                ],
                "explanation_en": "...",
                "explanation_vi": "...",
                "confidence": 0.95
            }
        ]
    }
    """
    try:
        bp_dict  = json.loads(exam_blueprint)
        st_dict  = json.loads(student_input)
        student_input_correct = { "success": True, "final_markdown": { "questions": [ { "question_number": "Bài I", "main_content": "", "sub_parts": [ { "part": "a", "content": r""" Ta có (1-2x)(x+5)=0 nên 1-2x=0 hoặc x+5=0. Từ đó suy ra x=1/2 hoặc x=-5. Vậy nghiệm của phương trình là x=1/2, x=-5. """ }, { "part": "b", "content": r""" Điều kiện xác định: x \ne 2, x \ne -2. \[ \frac{x+2}{x-2} - \frac{x-2}{2+x} = \frac{x^2+16}{x^2-4} \] Vì 2+x=x+2 nên ta quy đồng: \[ \frac{(x+2)^2}{(x-2)(x+2)} - \frac{(x-2)^2}{(x-2)(x+2)} = \frac{x^2+16}{(x-2)(x+2)} \] Suy ra: \[ (x+2)^2 - (x-2)^2 = x^2+16 \] Khai triển: \[ x^2+4x+4 - (x^2-4x+4) = x^2+16 \] Rút gọn: \[ 8x = x^2+16 \] Suy ra: \[ x^2-8x+16=0 \] \[ (x-4)^2=0 \Rightarrow x=4 \] Giá trị này thỏa mãn điều kiện. Vậy nghiệm của phương trình là x=4. """ } ] } ] } }
        result   = grade_full_pipeline(
            exam_blueprint=bp_dict,
            student_input=student_input_correct,
            debug=debug,
        )
        return {"success": True, **result}
    except json.JSONDecodeError as e:
        return {"success": False, "error": f"JSON parse error: {e}"}
    except ValueError as e:
        return {"success": False, "error": f"Blueprint validation error: {e}"}
    except Exception as e:
        return {"success": False, "error": str(e)}


In [ ]:
# ── CELL 14 — Khởi động server ───────────────────────────────────
nest_asyncio.apply()
public_url = ngrok.connect(8000)
print(f"🚀 API online tại: {public_url.public_url}")
print(f"📖 Swagger UI:      {public_url.public_url}/docs")
print()
print("Endpoints:")
print(f"  POST {public_url.public_url}/api/v1/clean_image    — Làm sạch ảnh")
print(f"  POST {public_url.public_url}/api/v1/ocr            — OCR ảnh → markdown")
print(f"  POST {public_url.public_url}/api/v1/fix_vn_and_struct — Sửa lỗi + tách JSON")
print(f"  POST {public_url.public_url}/api/v1/grade          — Chấm điểm")
print(f"  POST {public_url.public_url}/api/v1/ocr_and_grade  — Pipeline đầy đủ 1 call")

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()